In [3]:
import pandas as pd
import numpy as np
import torch
import os
path ='/n/data1/hsph/biostat/celehs/lab/hat127/GAME_0527/'
os.chdir(path+'/src')

from config import config

config['path'] = '/n/data1/hsph/biostat/celehs/lab/hat127/GAME_0527/'

In [11]:
def concordance_index(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    assert len(a) == len(b)

    n = len(a)
    concordant = 0
    discordant = 0
    tied = 0

    for i in range(n):
        for j in range(i + 1, n):
            if b[i] == b[j]:
                continue  # No order between same outcomes
            if b[i] > b[j]:
                pred_diff = a[i] - a[j]
            else:
                pred_diff = a[j] - a[i]

            if pred_diff > 0:
                concordant += 1
            elif pred_diff < 0:
                discordant += 1
            else:
                tied += 1

    total = concordant + discordant + tied
    if total == 0:
        return np.nan  # Undefined if all b are equal

    return (concordant + 0.5 * tied) / total

def read_and_compute(code_list, name_list,note='0519'):
    C_index_list = []
    for code in code_list:
        C_index_list2 = []
        colnames = [name + '_cos' for name in name_list]
        score_file = pd.read_csv(f"{config['path']}/supp_code/feature_selection/score_all/GPT4_ans_{code}_{note}.csv")
        score = score_file[colnames]
        for i in range(len(name_list)):
            C_index = concordance_index(score_file['gpt4'], score.iloc[:,i])
            C_index_list2.append(C_index)
        C_index_list.append(C_index_list2)
    return pd.DataFrame(C_index_list, index=code_list, columns=name_list)

In [7]:
# RESULT0610

# x_all = [x4, x7, x5, x1, x6, x3, x2, BIOBERT, PUBMED, x8, x9, x10, x11, GAME  ]
name_list = ['BCH', 'BDX','Duke','MGB','MIMIC','UPMC','VA', 'BBERT', 'PBERT', 'SBERT', 'CODER', 'BGE', 'OpenAI', 'GAME' ]
code_list = ["PheCode:428.1", "PheCode:296.2", "PheCode:714", "PheCode:290.11", "PheCode:250.1", "PheCode:250.2", "PheCode:555.1", "PheCode:555.2", 'PheCode:714.1']

read_and_compute(code_list, name_list, note='RESULT0610').T  # take the union!!  # 40

,PheCode:428.1,PheCode:296.2,PheCode:714,PheCode:290.11,PheCode:250.1,PheCode:250.2,PheCode:555.1,PheCode:555.2,PheCode:714.1
BCH,0.641818,0.580996,0.545987,0.539704,0.498260,0.514593,0.584501,0.563872,0.594758
BDX,0.637856,0.620982,0.632544,0.591203,0.567090,0.571748,0.585970,0.606743,0.604709
Duke,0.599265,0.535906,0.525827,0.539704,0.472649,0.497978,0.493313,0.498470,0.498873
MGB,0.698772,0.659591,0.644910,0.574410,0.554537,0.595907,0.611885,0.606778,0.638678
MIMIC,0.584041,0.498618,0.472550,0.521276,0.477905,0.505770,0.496135,0.495569,0.399711
UPMC,0.631855,0.584680,0.595065,0.576223,0.587006,0.609072,0.534885,0.542992,0.587102
VA,0.702683,0.628051,0.632063,0.569442,0.580921,0.602683,0.566127,0.564440,0.621423
BBERT,0.462884,0.397071,0.417021,0.481407,0.424313,0.431658,0.410659,0.474793,0.538624
PBERT,0.408402,0.442255,0.425851,0.476549,0.438421,0.449259,0.442306,0.433205,0.472727
SBERT,0.634786,0.408179,0.646482,0.587020,0.557285,0.532079,0.515879,0.497855,0.508141


In [8]:
emb_list = ['base_a_', 'base_b_','base_d_','base_e_','base_f_','base_g_','base_h_','GAME']

code_list = ["PheCode:428.1", "PheCode:296.2", "PheCode:714", "PheCode:290.11", "PheCode:250.1", "PheCode:250.2", "PheCode:555.1", "PheCode:555.2"]

print(read_and_compute(code_list, emb_list, note='ABLATION0610_2'))  # take the union!!  # 40

                 base_a_   base_b_   base_d_   base_e_   base_f_   base_g_  \
PheCode:428.1   0.643258  0.609572  0.523774  0.636240  0.639083  0.649677   
PheCode:296.2   0.665539  0.671190  0.453078  0.657882  0.646205  0.616251   
PheCode:714     0.572840  0.558913  0.556894  0.617454  0.624676  0.648127   
PheCode:290.11  0.601692  0.550962  0.524911  0.690488  0.710039  0.716862   
PheCode:250.1   0.626548  0.588834  0.537695  0.622172  0.627996  0.655356   
PheCode:250.2   0.626095  0.604779  0.463350  0.589296  0.613109  0.619013   
PheCode:555.1   0.598894  0.578151  0.527226  0.630931  0.635896  0.636697   
PheCode:555.2   0.638958  0.517274  0.541425  0.637280  0.660932  0.669370   

                 base_h_      GAME  
PheCode:428.1   0.614147  0.602950  
PheCode:296.2   0.654125  0.653002  
PheCode:714     0.617230  0.604480  
PheCode:290.11  0.708816  0.699413  
PheCode:250.1   0.635295  0.626231  
PheCode:250.2   0.621754  0.617155  
PheCode:555.1   0.636657  0.634174  
P